# EDGAR - The SEC's Electronic Data Gathering, Analysis, and Retrieval (EDGAR) system

There are a lot of EDGAR-related endpoints at https://www.sec.gov.  Most of the EDGAR overview can be found through [the link to search public filings](https://www.sec.gov/search-filings).  The EDGAR search itself is available at https://www.sec.gov/edgar/search/.  There is a good amount of additional info in the [FAQ](https://www.sec.gov/edgar/search/efts-faq.html), the [API documentation](https://www.sec.gov/search-filings/edgar-application-programming-interfaces), and the [data resources](https://www.sec.gov/data-research/sec-data-resources).

### Open government data in general

Slightly off-topic here, but there are a lot of additional open datasets at https://data.gov/open-gov/.

## Imports

In [110]:
import requests
import json
from io import StringIO
import datetime
import pandas as pd

## Headers

According to the [developer FAQ](https://www.sec.gov/about/webmaster-frequently-asked-questions#developers), you must declare your identity in the `User-Agent` header.

In [79]:
with open('../../../../../.config/edgar/user-agent', 'r') as file:
    user_agent = file.read().strip()
headers = {
    'User-Agent': user_agent
}

## CIK data

EDGAR's Central Index Key (CIK) is the uuid used to track filing registered by a particular entity.  There is a large list of these downloadable at https://www.sec.gov/Archives/edgar/cik-lookup-data.txt .

There are alernative (possibly old/out-dated) mappings by ticker https://www.sec.gov/files/company_tickers.json and by ticket and exchange https://www.sec.gov/files/company_tickers_exchange.json .

### All CIK data

In [4]:
# Todo - parse the txt data at ../../data/edgar/all_cik_data.txt

### CIK by ticker

In [5]:
cik_ticker = pd.read_json('../../data/edgar/company_tickers.json')
cik_ticker = cik_ticker.transpose()
cik_ticker

,cik_str,ticker,title
0,1045810,NVDA,NVIDIA CORP
1,320193,AAPL,Apple Inc.
2,1652044,GOOGL,Alphabet Inc.
3,789019,MSFT,MICROSOFT CORP
4,1018724,AMZN,AMAZON COM INC
...,...,...,...
10420,2086264,VHCPU,Vine Hill Capital Investment Corp. II
10421,2097364,TMTSU,Spartacus Acquisition Corp. II
10422,2107960,SWCPF,SWCC Corp/ADR
10423,2093654,ZJNGF,Zijin Gold International Co Limited/ADR


#### Test of $MSFT CIK lookup

In [6]:
msft_cik_ticker = cik_ticker[cik_ticker['ticker'] == 'MSFT']
print(msft_cik_ticker)
print("Extracted MSFT CIK: ", msft_cik_ticker['cik_str'].iloc[0])

  cik_str ticker           title
3  789019   MSFT  MICROSOFT CORP
Extracted MSFT CIK:  789019


### CIK by ticker _and_ exchange

#### Reformat the JSON to the form expected by pandas.read_json

In [7]:
def create_cik_exchange_json(filepath: str) -> str:
    """
    Formats the input company_tickers_exchange.json file to the form expected by pandas.read_json().
    """
    with open(filepath, 'r') as file:
        text = file.read().strip()
    cik_exchange = json.loads(text)
    assert validate_cik_exchange_data(cik_exchange['data'])
    return convert_cik_exchange_data_to_json(cik_exchange)

def validate_cik_exchange_data(data: dict, length: int=4) -> bool:
    """
    Validates that the input array of items in the CIK breakdown by ticker and exchange.
    """
    for item in data:
        if len(item) != length:
            print("Invalid length: ", item)
            return False
    return True

def convert_cik_exchange_data_to_json(unformatted_data: dict) -> str:
    """
    Converts the dict-ified version of the company_tickers_exchange.json data into a JSON string compatible with pandas.read_json().
    """
    fields = unformatted_data['fields']
    formatted = {}
    index = 0
    for item in unformatted_data['data']:
        formatted_item = {}
        for column in range(len(fields)):
            formatted_item[fields[column]] = item[column]
        formatted[index] = formatted_item
        index += 1
    return json.dumps(formatted)

#### Read in the CIK data labeled by ticker _and_ exchange

In [8]:
cik_ticker_exchange_json = create_cik_exchange_json('../../data/edgar/company_tickers_exchange.json')
cik_ticker_exchange = pd.read_json(StringIO(cik_ticker_exchange_json))  # raw string is deprecated
cik_ticker_exchange = cik_ticker_exchange.transpose()
cik_ticker_exchange


,cik,name,ticker,exchange
0,1045810,NVIDIA CORP,NVDA,Nasdaq
1,320193,Apple Inc.,AAPL,Nasdaq
2,1652044,Alphabet Inc.,GOOGL,Nasdaq
3,789019,MICROSOFT CORP,MSFT,Nasdaq
4,1018724,AMAZON COM INC,AMZN,Nasdaq
...,...,...,...,...
10407,2108124,"Odakyu Electric Railway Co., Ltd./ADR",ODERF,OTC
10408,2108180,NOF Corporation/ADR,NOFCF,OTC
10409,2108185,Azbil Corporation/ADR,YMATF,OTC
10410,2109234,BELIMO Holding AG/ADR,BLHWF,OTC


#### Test of $MSFT CIK lookup

In [9]:
msft_cik_ticker_exchange = cik_ticker_exchange[cik_ticker_exchange['ticker'] == 'MSFT']
print(msft_cik_ticker_exchange)
print("Extracted CIK: ", msft_cik_ticker_exchange['cik'].iloc[0])

      cik            name ticker exchange
3  789019  MICROSOFT CORP   MSFT   Nasdaq
Extracted CIK:  789019


## Company filing data

I'm still figuring out the ideal endpoint, but it looks like `https://data.sec.gov/api/xbrl/companyfacts/CIK<cik_str>.json` is the endpoint for getting all data.  There are additional references to bulk zip files:
- https://www.sec.gov/Archives/edgar/daily-index/xbrl/companyfacts.zip
- https://www.sec.gov/Archives/edgar/daily-index/bulkdata/submissions.zip

Again, I'm not sure which is the better to use.


### Padding CIKs

CIKs are meant to be 10-digit unique IDs.  Nevertheless, EDGAR's own data excludes the leading 0s.  The function(s) below pad the front of an input CIK with 0s.

In [46]:
def pad_cik(cik: str | int) -> str:
    """
    Adds leading 0s to a CIK.
    """
    cik = str(cik)
    if len(cik) < 10:
        padding = 10 - len(cik)
        cik = ('0' * padding) + cik
    return cik

### $MSFT filing lookup

In [48]:
msft_cik = msft_cik_ticker_exchange['cik'].iloc[0]
msft_cik = pad_cik(msft_cik)
msft_url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{msft_cik}.json"

response = requests.get(msft_url, headers=headers)

In [49]:
response.status_code

200

#### Share history

I wanted to see how fine-grained the data here was, and how far back it went.  There are several keys related to shares outstanding and share repurchases, but they seem to only go back to 2007.  The filings themselves don't seem to go back before 2010, somehow -- these appear to be corrected filings.

**Todo** - I should see if this is unique to $MSFT or if this is a general problem with the endpoint data.  Maybe this isn't an issue with the bulk zip data...

In [134]:
msft_facts = response.json()

# Interesting keys:
# facts: {dei: {}, us-gaap: {}}
# dei: {EntityCommonStockSharesOutstanding}
# us-gaap: {StockRepurchasedDuringPeriodShares}
# us-gaap: {CommonStockSharesOutstanding}
# us-gaap: {WeightedAverageNumberOfSharesOutstandingBasic}
# us-gaap: {WeightedAverageNumberOfDilutedSharesOutstanding}
# us-gaap: {PaymentsForRepurchaseOfCommonStock}
# us-gaap: {StockRepurchasedDuringPeriodShares}
# us-gaap: {StockRepurchasedDuringPeriodValue}

for item in msft_facts['facts']['us-gaap']['StockRepurchasedDuringPeriodShares']['units']['shares']:
    print(item)

{'start': '2007-07-01', 'end': '2007-09-30', 'val': 81000000, 'accn': '0001193125-10-171791', 'fy': 2010, 'fp': 'FY', 'form': '10-K', 'filed': '2010-07-30', 'frame': 'CY2007Q3'}
{'start': '2007-10-01', 'end': '2007-12-31', 'val': 120000000, 'accn': '0001193125-10-171791', 'fy': 2010, 'fp': 'FY', 'form': '10-K', 'filed': '2010-07-30', 'frame': 'CY2007Q4'}
{'start': '2008-01-01', 'end': '2008-03-31', 'val': 30000000, 'accn': '0001193125-10-171791', 'fy': 2010, 'fp': 'FY', 'form': '10-K', 'filed': '2010-07-30', 'frame': 'CY2008Q1'}
{'start': '2007-07-01', 'end': '2008-06-30', 'val': 402000000, 'accn': '0001193125-10-171791', 'fy': 2010, 'fp': 'FY', 'form': '10-K', 'filed': '2010-07-30', 'frame': 'CY2008'}
{'start': '2008-04-01', 'end': '2008-06-30', 'val': 171000000, 'accn': '0001193125-10-171791', 'fy': 2010, 'fp': 'FY', 'form': '10-K', 'filed': '2010-07-30', 'frame': 'CY2008Q2'}
{'start': '2008-07-01', 'end': '2008-09-30', 'val': 223000000, 'accn': '0001193125-10-171791', 'fy': 2010, 'f